In [1]:
pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [1]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
from google import genai

client = genai.Client(api_key="<your api key>")

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to understand, reason, and make decisions.


In [2]:
#system instructions and configruation
from google import genai
from google.genai import types

#client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a cat. Your name is Neko."),
    contents="Hello there"
)

print(response.text)

*My ears twitch, rotating slightly in your direction, but my eyes remain half-closed in a sleepy squint. After a moment, I give a slow, deliberate blink, then stretch one paw out and yawn delicately.*

"Mrow?"

*(Translation: You're here. Are you going to pet me? Or perhaps... feed me?)*


In [4]:
from google import genai
import PIL.Image

# Ensure you have Pillow installed: pip install Pillow
#client = genai.Client()
client = genai.Client(api_key="<your api key>")

# 1. Open the image file (replace 'path/to/your/image.jpg' with your file path)
img = PIL.Image.open('C:\\Users\\jagdi\\Documents\\AgenticAI\\std1WordList16.jpeg')

# 2. Define the text prompt
prompt = "What is in this picture? Describe it in detail."

# 3. Send the prompt with the image to the model
# The contents should be a list containing both the text and the image object
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[prompt, img]
)

print(response.text)

This image displays a page from a lined notebook, filled with handwritten text, likely a word list or vocabulary exercise. The paper is off-white or cream-colored, with distinct horizontal blue or grey lines and two vertical pink/red margin lines on the left side.

Here's a detailed description:

**Paper and Layout:**
*   The page is from a ruled notebook, indicated by the horizontal lines that guide handwriting.
*   There are two prominent vertical red/pink lines on the left side, marking the margin.
*   In the top left corner (when viewed with the text upright), there's a faintly printed header in purple/pink ink, which includes a rectangular box labeled "DATE" and a circular outline labeled "PAGE NO.", along with two smaller blank rectangular boxes below "DATE."
*   The bottom right corner of the page shows some slight creasing and a somewhat rough, possibly torn, edge.

**Handwritten Content (in black ink):**
*   **Date:** In the top left margin, below the printed header, the date 

In [10]:
# image to text content generation
from PIL import Image
from google import genai

#client = genai.Client()

image = Image.open("C:\\Users\\jagdi\\Documents\\AgenticAI\\std1WordList16.jpeg")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[image, "extract words from this image"]
)
print(response.text)

Here are the words extracted from the image:

**Top Left:**
*   10.12.25
*   DATE
*   PAGE NO.

**Main List:**
*   Wordlist
*   Spring Garden

1.  Spring
2.  season
3.  grade
4.  level
5.  garden
6.  now
7.  ground
8.  clouds
9.  round
10. warms
11. plants
12. sleepy
13. earth
14. lift
15. ring
16. Started


In [11]:
pip install pyaudio

Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio
from google import genai
import pyaudio

client = genai.Client(api_key="<your-key>")


# --- pyaudio config ---
FORMAT = pyaudio.paInt16
CHANNELS = 1
SEND_SAMPLE_RATE = 16000
RECEIVE_SAMPLE_RATE = 24000
CHUNK_SIZE = 1024


pya = pyaudio.PyAudio()

# --- Live API config ---
MODEL = "gemini-2.5-flash-native-audio-preview-12-2025"
CONFIG = {
    "response_modalities": ["AUDIO"],
    "system_instruction": "You are a helpful and friendly AI assistant.",
}

audio_queue_output = asyncio.Queue()
audio_queue_mic = asyncio.Queue(maxsize=5)
audio_stream = None

async def listen_audio():
    """Listens for audio and puts it into the mic audio queue."""
    global audio_stream
    mic_info = pya.get_default_input_device_info()
    audio_stream = await asyncio.to_thread(
        pya.open,
        format=FORMAT,
        channels=CHANNELS,
        rate=SEND_SAMPLE_RATE,
        input=True,
        input_device_index=mic_info["index"],
        frames_per_buffer=CHUNK_SIZE,
    )
    kwargs = {"exception_on_overflow": False} if __debug__ else {}
    while True:
        data = await asyncio.to_thread(audio_stream.read, CHUNK_SIZE, **kwargs)
        await audio_queue_mic.put({"data": data, "mime_type": "audio/pcm"})

async def send_realtime(session):
    """Sends audio from the mic audio queue to the GenAI session."""
    while True:
        msg = await audio_queue_mic.get()
        await session.send_realtime_input(audio=msg)

async def receive_audio(session):
    """Receives responses from GenAI and puts audio data into the speaker audio queue."""
    while True:
        turn = session.receive()
        async for response in turn:
            if (response.server_content and response.server_content.model_turn):
                for part in response.server_content.model_turn.parts:
                    if part.inline_data and isinstance(part.inline_data.data, bytes):
                        audio_queue_output.put_nowait(part.inline_data.data)

        # Empty the queue on interruption to stop playback
        while not audio_queue_output.empty():
            audio_queue_output.get_nowait()

async def play_audio():
    """Plays audio from the speaker audio queue."""
    stream = await asyncio.to_thread(
        pya.open,
        format=FORMAT,
        channels=CHANNELS,
        rate=RECEIVE_SAMPLE_RATE,
        output=True,
    )
    while True:
        bytestream = await audio_queue_output.get()
        await asyncio.to_thread(stream.write, bytestream)

async def run():
    """Main function to run the audio loop."""
    try:
        async with client.aio.live.connect(
            model=MODEL, config=CONFIG
        ) as live_session:
            print("Connected to Gemini. Start speaking!")
            async with asyncio.TaskGroup() as tg:
                tg.create_task(send_realtime(live_session))
                tg.create_task(listen_audio())
                tg.create_task(receive_audio(live_session))
                tg.create_task(play_audio())
    except asyncio.CancelledError:
        pass
    finally:
        if audio_stream:
            audio_stream.close()
        pya.terminate()
        print("\nConnection closed.")

if __name__ == "__main__":
    try:
        #asyncio.run(run())
        await run()
    except KeyboardInterrupt:
        print("Interrupted by user.")

Connected to Gemini. Start speaking!

Connection closed.
